In [1]:

import re
import torch
from torch.utils.data import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix  # Aseguramos confusion_matrix
from razdel import sentenize
import nltk
from isanlp.pipeline_common import PipelineCommon
from transformers import logging
from tqdm import tqdm
import torch.nn.functional as F
import warnings
import gc


# Suprimir warnings específicos
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
#mp.set_start_method('spawn', force=True)


In [2]:
def cargar_datos(archivo, etiqueta):
    with open(archivo, 'r', encoding='utf-8') as f:
        textos = f.readlines()
    return [(texto.strip(), etiqueta) for texto in textos]

In [3]:
def tokenizar_fuction(texto):
    """Tokeniza el texto eliminando puntuación y convirtiendo a minúsculas."""
    texto_preprocesado = re.sub(r'[^\w\s]', '', texto.lower())
    tokens = nltk.word_tokenize(texto_preprocesado)
    return tokens

In [4]:
class DataLoader_raw:
    def __init__(self, path1, path2):
        self.path1 = path1
        self.path2 = path2
    
    def __call__(self):
        datos_genero1 = cargar_datos(self.path1, 0)
        datos_genero2 = cargar_datos(self.path2, 1)
        datos = datos_genero1 + datos_genero2
        
        return {
            'datos_raw': datos,
        }

In [5]:
class SentenceCleaner:
    def __init__(self, min_length_threshold=6):
        """Inicializa el procesador con un tokenizador y un umbral de longitud mínima."""

        self.min_length_threshold = min_length_threshold
    
    def _clean_and_split_text(self, texto):
        """
        Limpia y divide un texto en oraciones procesadas.
        
        Args:
            texto (str): Texto crudo a procesar.
        
        Returns:
            list: Lista de oraciones limpias.
        """
        # Paso 1: Marcar puntos para evitar fusiones no deseadas
        texto_limpio = re.sub(r'\.,', '. Ok999999999 ,', texto)  # Caso 1: Punto seguido de coma
        texto_limpio = re.sub(r'\.;', '. Ok999999999 ', texto_limpio)  # Caso 2: Punto seguido de punto y coma
        texto_limpio = re.sub(r'\. ([a-zа-я])', r'. Ok999999999 \1', texto_limpio)  # Caso 3: Punto seguido de minúscula
        texto_limpio = re.sub(r'(\w)([А-Я])', r'\1. \2', texto_limpio)  # Caso 4: Minúscula seguida de mayúscula
        
        #creo que hay que agregar una nueva condivion  дл.,0 
        #", 2-3 см шир.",0 
        #", 1 см шир.",0 si hay un punto uego numeros quitar el punto
        
        # Paso 2: Eliminar corchetes y dividir en oraciones
        texto_sin_corchetes = re.sub(r'\[.*?\]', '', texto_limpio).strip()
        oraciones = [oracion.text for oracion in sentenize(texto_sin_corchetes)]
        
        # Paso 3: Restaurar espacios y limpiar marcadores temporales
        oraciones_limpias = [re.sub(r'\s*Ok999999999', ' ', oracion).strip() for oracion in oraciones]
        
        return oraciones_limpias
    
    def _process_sentences(self, datos, target_list):
      
        for texto, etiqueta in datos:
            oraciones = self._clean_and_split_text(texto)
            for oracion in oraciones:
                # Filtrar oraciones cortas basadas en el número de tokens
                tokens = tokenizar_fuction(oracion)
                if len(tokens) >= self.min_length_threshold:####modificacion aqui
                    target_list.append((oracion, etiqueta))
    
    def __call__(self, data_raw):
        datos_procesados = []
        self._process_sentences(data_raw, datos_procesados)
        
        return {
            'datos_procesados': datos_procesados,
        }
#         return {
#             'datos_procesados': data_raw,
#         }

In [6]:
class TextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

In [7]:
datasets = [
    #deberia adicionar el set original?
#         {
#             'path1': '../dataset/сокращение по частотности/1б_Изъяты лексемы с частотой выше 100.txt',
#             'path2': '../dataset/сокращение по частотности/2б_Изъяты лексемы с частотой выше 100.txt',
#             'name': 'Изъяты лексемы с частотой выше 100',
#             'type': 'freq',
#             'freq': 100
#         },
#         {
#             'path1': '../dataset/сокращение по частотности/1в_Изъяты лексемы с частотой выше 49.txt',
#             'path2': '../dataset/сокращение по частотности/2в_Изъяты лексемы с частотой выше 49.txt',
#             'name': 'Изъяты лексемы с частотой выше 49',
#             'type': 'freq',
#             'freq': 49
#         },
#         {
#             'path1': '../dataset/сокращение по частотности/1г_Изъяты лексемы с частотой выше 29.txt',
#             'path2': '../dataset/сокращение по частотности/2г_Изъяты лексемы с частотой выше 29.txt',
#             'name': 'Изъяты лексемы с частотой выше 29',
#             'type': 'freq',
#             'freq': 29
#         },
#         {
#             'path1': '../dataset/сокращение по частотности/1д_Изъяты лексемы с частотой выше 9.txt',
#             'path2': '../dataset/сокращение по частотности/2д_Изъяты лексемы с частотой выше 9.txt',
#             'name': 'Изъяты лексемы с частотой выше 9',
#             'type': 'freq',
#             'freq': 9
#         },
#         {
#             'path1': '../dataset/сокращение по частотности/1е_Изъяты лексемы с частотой выше 5.txt',
#             'path2': '../dataset/сокращение по частотности/2е_Изъяты лексемы с частотой выше 5.txt',
#             'name': 'Изъяты лексемы с частотой выше 5',
#             'type': 'freq',
#             'freq': 5
#         },
#         {
#             'path1': '../dataset/сокращение по частотности/1ё_Изъяты лексемы с частотой выше 3.txt',
#             'path2': '../dataset/сокращение по частотности/2ё_Изъяты лексемы с частотой выше 3.txt',
#             'name': 'Изъяты лексемы с частотой выше 3',
#             'type': 'freq',
#             'freq': 3
#         },

#             {
#             'path1': '../dataset/Сокращение по частям речи/Без прилагательных первый жанр.txt',
#             'path2': '../dataset/Сокращение по частям речи/Без прилагательных второй жанр.txt',
#             'name': 'Без прилагательных первый-второй жанр',#litle correction
#             'type': 'pos',
#             'freq': None
#             },
            {
            'path1': '../dataset/Первый_жанр_исходная.txt',
            'path2': '../dataset/Второй_жанр_исходная.txt',
            'name': '1.Первый жанр исходная выборка',# este el el orignal
            'type': 'pos',
            'freq': None

            },
#           {
#             'path1': '../dataset/Сокращение по частям речи/1.Первый жанр исходная выборка.txt',
#             'path2': '../dataset/Второй_жанр_исходная.txt',
#             'name': '1.Первый жанр исходная выборка',# este el el orignal
#             'type': 'pos',
#             'freq': None

#         },
#         {
#             'path1': '../dataset/Сокращение по частям речи/2.Первый жанр без клауз, включающих наречия.txt',
#             'path2': '../dataset/Второй_жанр_исходная.txt',
#             'name': '2.Первый жанр без клауз, включающих наречия',
#             'type': 'pos',
#             'freq': None
#         },
#         {
#             'path1': '../dataset/Сокращение по частям речи/3.Первый жанр без клауз, включающих глаголы.txt',
#             'path2': '../dataset/Второй_жанр_исходная.txt',
#             'name': '3.Первый жанр без клауз, включающих глаголы',
#             'type': 'pos',
#             'freq': None
#         },
#         {
#             'path1': '../dataset/Сокращение по частям речи/4.Первый жанр без клауз, включающих глаголы и наречия.txt',
#             'path2': '../dataset/Второй_жанр_исходная.txt',
#             'name': '4.Первый жанр без клауз, включающих глаголы и наречия',
#             'type': 'pos',
#             'freq': None
#         },
#         {
#             'path1': '../dataset/Сокращение по частям речи/5.без клауз, включающих местоимения.txt',
#             'path2': '../dataset/Второй_жанр_исходная.txt',
#             'name': '5.без клауз, включающих местоимения.txt',
#             'type': 'pos',
#             'freq': None
#         },
#         {
#             'path1': '../dataset/Сокращение по частям речи/6.без слов функциональных.txt',
#             'path2': '../dataset/Второй_жанр_исходная.txt',
#             'name': '6.без слов функциональных.txt',
#             'type': 'pos',
#             'freq': None
#         },
    
    ]

In [8]:
# zero_shot_models = [
#     {'model': 'gpt2',
#      'name': 'gpt2',
#      'type': 'gpt'},  # GPT model
#     {'model': 'facebook/opt-125m',
#      'name': 'facebook/opt-125m',
#      'type': 'gpt'},  # GPT model
#     {'model': 'sberbank-ai/rugpt3small_based_on_gpt2',
#      'name': 'rugpt3small',
#      'type': 'gpt'}, # GPT model (ruso)
#     {'model': 'sberbank-ai/rugpt3medium_based_on_gpt2',
#      'name': 'rugpt3medium',
#      'type': 'gpt'},
# ]

zero_shot_models = [
#     {'model': 'gpt2',
#      'name': 'gpt2',
#      'type': 'gpt'},  # GPT model
#     {'model': 'facebook/opt-125m',
#      'name': 'facebook/opt-125m',
#      'type': 'gpt'},  # GPT model
    {'model': 'sberbank-ai/rugpt3small_based_on_gpt2',
     'name': 'rugpt3small',
     'type': 'gpt'}, # GPT model (ruso)
#     {'model': 'sberbank-ai/rugpt3medium_based_on_gpt2',
#      'name': 'rugpt3medium',
#      'type': 'gpt'},

    # Modelos GPT más pequeños
#     {'model': 'gpt2',
#      'name': 'gpt2-small', # Nombre para identificar la versión pequeña de GPT-2
#      'type': 'gpt'},
#     {'model': 'openai-community/gpt2-medium', # A veces gpt2 sin especificación apunta a la versión pequeña, pero para ser explícito
#      'name': 'gpt2-medium',
#      'type': 'gpt'},
#     {'model': 'EleutherAI/gpt-neo-125m', # Un modelo GPT-like pequeño y popular de EleutherAI
#      'name': 'gpt-neo-125m',
#      'type': 'gpt'},
#     {'model': 'PygmalionAI/pygmalion-125m', # Otro modelo pequeño basado en GPT-Neo
#      'name': 'pygmalion-125m',
#      'type': 'gpt'},
#     {'model': 'google/gemma-2b', # Un modelo relativamente pequeño y eficiente de Google
#      'name': 'gemma-2b',
#      'type': 'gpt'}, # Aunque es un modelo de Google, su arquitectura es transformer similar a GPT
#     {'model': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0', # Un ejemplo de TinyLlama, ideal para eficiencia
#      'name': 'tinyllama-1.1b-chat',
#      'type': 'gpt'},
    # Podrías agregar más aquí según tus necesidades
]

In [9]:
prompts = [
    {
        'name': 'Prompt_Original',
        'template': (
            "Классифицируй следующий текст как 'определение' или 'описание'. "
            "Определение объясняет значение термина или понятия, часто в формальном стиле. "
            "Описание рисует картину объекта, сцены или явления с деталями.\n\n"
            "Примеры:\n"
            "Текст: 'Философия — это наука, изучающая общие законы развития природы, общества и мышления.'\n"
            "Ответ: определение\n\n"
            "Текст: 'Демократия — это форма правления, при которой власть принадлежит народу.'\n"
            "Ответ: определение\n\n"
            "Текст: 'Годовалые побеги утолщенные, короткие, темно-бурые, при сушке чернеющие, серовато-шерстисто-опушенные.'\n"
            "Ответ: описание\n\n"
            "Текст: 'Река сверкала под лучами солнца, а берега утопали в зелени.'\n"
            "Ответ: описание\n\n"
            f"Текст: {{text}}\n\n"
            "Ответ: "
        )
    },
    {
        'name': 'Prompt_Simple',
        'template': (
            "Определи, является ли текст 'определением' или 'описанием'. "
            "Определение объясняет, что такое объект или понятие. "
            "Описание описывает внешний вид или характеристики объекта.\n\n"
            f"Текст: {{text}}\n\n"
            "Ответ: "
        )
    },
    {
        'name': 'Prompt_Detailed',
        'template': (
            "Классифицируй текст как 'определение' или 'описание'. "
            "Определение — это формальное объяснение термина, понятия или процесса, обычно с четкой структурой. "
            "Описание — это детализированное изображение объекта, сцены или явления с акцентом на визуальные или сенсорные характеристики.\n\n"
            "Примеры:\n"
            "Текст: 'Биология — наука, изучающая живые организмы и их взаимодействие с окружающей средой.'\n"
            "Ответ: определение\n\n"
            "Текст: 'Лес был густым, с высокими соснами, покрытыми мхом, и запахом хвои.'\n"
            "Ответ: описание\n\n"
            f"Текст: {{text}}\n\n"
            "Ответ: "
        )
    },
        {
        'name': 'Prompt_Original_reazon',
        'template': (
            "Классифицируй следующий текст как 'определение' или 'описание'. "
            "Определение объясняет значение термина или понятия, часто в формальном стиле. "
            "Описание рисует картину объекта, сцены или явления с деталями.\n\n"
            "Примеры:\n"
            "Текст: 'Философия — это наука, изучающая общие законы развития природы, общества и мышления.'\n"
            "Ответ: Классификация: определение. Причина: Текст формально объясняет значение термина 'философия'.\n\n"
            "Текст: 'Демократия — это форма правления, при которой власть принадлежит народу.'\n"
            "Ответ: Классификация: определение. Причина: Текст четко определяет понятие 'демократия'.\n\n"
            "Текст: 'Годовалые побеги утолщенные, короткие, темно-бурые, при сушке чернеющие, серовато-шерстисто-опушенные.'\n"
            "Ответ: Классификация: описание. Причина: Текст детализирует внешние характеристики побегов.\n\n"
            "Текст: 'Река сверкала под лучами солнца, а берега утопали в зелени.'\n"
            "Ответ: Классификация: описание. Причина: Текст создает визуальную картину сцены.\n\n"
            f"Текст: {{text}}\n\n"
            "Ответ: Классификация: [определение/описание]. Причина: [объяснение, почему выбрана эта классификация]."
        )
    },
    {
        'name': 'Prompt_Simple_reazon',
        'template': (
            "Определи, является ли текст 'определением' или 'описанием'. "
            "Определение объясняет, что такое объект или понятие. "
            "Описание описывает внешний вид или характеристики объекта.\n\n"
            f"Текст: {{text}}\n\n"
            "Ответ: Классификация: [определение/описание]. Причина: [объяснение, почему выбрана эта классификация]."
        )
    },
    {
        'name': 'Prompt_Detailed_reazon',
        'template': (
            "Классифицируй текст как 'определение' или 'описание'. "
            "Определение — это формальное объяснение термина, понятия или процесса, обычно с четкой структурой. "
            "Описание — это детализированное изображение объекта, сцены или явления с акцентом на визуальные или сенсорные характеристики.\n\n"
            "Примеры:\n"
            "Текст: 'Биология — наука, изучающая живые организмы и их взаимодействие с окружающей средой.'\n"
            "Ответ: Классификация: определение. Причина: Текст формально объясняет, что такое биология.\n\n"
            "Текст: 'Лес был густым, с высокими соснами, покрытыми мхом, и запахом хвои.'\n"
            "Ответ: Классификация: описание. Причина: Текст создает визуальное и сенсорное изображение леса.\n\n"
            f"Текст: {{text}}\n\n"
            "Ответ: Классификация: [определение/описание]. Причина: [объяснение, почему выбрана эта классификация]."
        )
    }
]



In [10]:
def zero_shot_classification_batch(texts, model, tokenizer, prompts, device='cuda' if torch.cuda.is_available() else 'cpu', batch_size=4):
    """Clasificación zero-shot en lotes usando logits para múltiples prompts."""
    results_per_prompt = {prompt['name']: [] for prompt in prompts}
    label_tokens = tokenizer(["определение", "описание"], return_tensors="pt", padding=True).input_ids[:, 0].to(device)
    
    for prompt in prompts:
        prompt_name = prompt['name']
        prompt_template = prompt['template']
        predictions = []
        
        for i in tqdm(range(0, len(texts), batch_size), desc=f"Clasificando con {prompt_name}"):
            batch_texts = texts[i:i + batch_size]
            batch_prompts = [prompt_template.format(text=text) for text in batch_texts]
            inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
            
            with torch.no_grad():
                outputs = model(**inputs)
                logits = outputs.logits[:, -1, :]
                probs = F.softmax(logits[:, label_tokens], dim=-1)
                pred_labels = probs.argmax(dim=-1).cpu().numpy()
                predictions.extend(pred_labels)
        
        predictions = [0 if p != 1 else 1 for p in predictions]
        results_per_prompt[prompt_name] = predictions
    
    return results_per_prompt



In [11]:
def evaluate_zero_shot(test_loader, model_name, prompts, batch_size=8):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
    model.config.pad_token_id = tokenizer.pad_token_id
    model.eval()

    texts = [text for text, _ in test_loader]
    true_labels = [label for _, label in test_loader]
    true_labels = [0 if l != 1 else 1 for l in true_labels]
    
    results_per_prompt = zero_shot_classification_batch(
        texts,
        model,
        tokenizer,
        prompts,
        device,
        batch_size
    )
    
    metrics_per_prompt = {}
    for prompt_name, pred_labels in results_per_prompt.items():
        pred_labels = [0 if p != 1 else 1 for p in pred_labels]
        accuracy = accuracy_score(true_labels, pred_labels)
        precision, recall, f1, _ = precision_recall_fscore_support(
            true_labels,
            pred_labels,
            average='binary',
            zero_division=0
        )
        cm = confusion_matrix(true_labels, pred_labels, labels=[0, 1])
        metrics_per_prompt[prompt_name] = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'confusion_matrix': cm
        }
    
    return metrics_per_prompt






In [12]:
def main():
    results = []
    
    for model in zero_shot_models:
        for dataset in datasets:
            ppl = PipelineCommon([
                (DataLoader_raw(dataset['path1'], dataset['path2']), [], {'datos_raw': 'datos_raw'}),
                (SentenceCleaner(), ['datos_raw'], {'datos_procesados': 'datos_procesados'}),
            ])
            result = ppl()
            test_loader = result['datos_procesados']
            
            metrics_per_prompt = evaluate_zero_shot(
                test_loader,
                model['model'],
                prompts,
                batch_size=16
            )#aqui estaba el problema
            
            for prompt_name, metrics in metrics_per_prompt.items():
                results.append({
                    'dataset': dataset['name'],
                    'model': model['name'],
                    'prompt': prompt_name,
                    'metrics': metrics
                })
        
        torch.cuda.empty_cache()
        gc.collect()
    
    for res in results:
        print(f"Dataset: {res['dataset']}, Model: {res['model']}, Prompt: {res['prompt']}")
        print(f"Métricas: {res['metrics']}")
        print(res['metrics']['confusion_matrix'])
        print()

if __name__ == "__main__":
    main()


Clasificando con Prompt_Detailed_reazon: 100%|██████████| 54/54 [00:04<00:00, 11.04it/s]


Dataset: 1.Первый жанр исходная выборка, Model: rugpt3small, Prompt: Prompt_Original
Métricas: {'accuracy': 0.35689455388180763, 'precision': 0.3177966101694915, 'recall': 0.391644908616188, 'f1': 0.3508771929824561, 'confusion_matrix': array([[158, 322],
       [233, 150]])}
[[158 322]
 [233 150]]

Dataset: 1.Первый жанр исходная выборка, Model: rugpt3small, Prompt: Prompt_Simple
Métricas: {'accuracy': 0.6071842410196987, 'precision': 0.6122448979591837, 'recall': 0.3133159268929504, 'f1': 0.4145077720207254, 'confusion_matrix': array([[404,  76],
       [263, 120]])}
[[404  76]
 [263 120]]

Dataset: 1.Первый жанр исходная выборка, Model: rugpt3small, Prompt: Prompt_Detailed
Métricas: {'accuracy': 0.4275782155272306, 'precision': 0.41874084919472915, 'recall': 0.7467362924281984, 'f1': 0.5365853658536586, 'confusion_matrix': array([[ 83, 397],
       [ 97, 286]])}
[[ 83 397]
 [ 97 286]]

Dataset: 1.Первый жанр исходная выборка, Model: rugpt3small, Prompt: Prompt_Original_reazon
Métric

In [ ]:
import re
import torch
from torch.utils.data import Dataset # Although TextDataset is not directly used for classification, keeping it for completeness if other parts of your pipeline use it
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from razdel import sentenize # Assuming these are available from your previous setup
import nltk
from isanlp.pipeline_common import PipelineCommon # Assuming this is available
from tqdm import tqdm
import torch.nn.functional as F # Although no longer used for logits, useful for other tasks
import warnings
import gc

# Suppress specific warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# Ensure NLTK punkt tokenizer is downloaded
try:
    nltk.data.find('tokenizers/punkt')
except nltk.downloader.DownloadError:
    nltk.download('punkt')

# Define your prompts (as you already have them)
prompts = [
    {
        'name': 'Prompt_Original',
        'template': (
            "Классифицируй следующий текст как 'определение' или 'описание'. "
            "Определение объясняет значение термина или понятия, часто в формальном стиле. "
            "Описание рисует картину объекта, сцены или явления с деталями.\n\n"
            "Примеры:\n"
            "Текст: 'Философия — это наука, изучающая общие законы развития природы, общества и мышления.'\n"
            "Ответ: определение\n\n"
            "Текст: 'Демократия — это форма правления, при которой власть принадлежит народу.'\n"
            "Ответ: определение\n\n"
            "Текст: 'Годовалые побеги утолщенные, короткие, темно-бурые, при сушке чернеющие, серовато-шерстисто-опушенные.'\n"
            "Ответ: описание\n\n"
            "Текст: 'Река сверкала под лучами солнца, а берега утопали в зелени.'\n"
            "Ответ: описание\n\n"
            f"Текст: {{text}}\n\n"
            "Ответ: "
        )
    },
    {
        'name': 'Prompt_Simple',
        'template': (
            "Определи, является ли текст 'определением' или 'описанием'. "
            "Определение объясняет, что такое объект или понятие. "
            "Описание описывает внешний вид или характеристики объекта.\n\n"
            f"Текст: {{text}}\n\n"
            "Ответ: "
        )
    },
    {
        'name': 'Prompt_Detailed',
        'template': (
            "Классифицируй текст как 'определение' или 'описание'. "
            "Определение — это формальное объяснение термина, понятия или процесса, обычно с четкой структурой. "
            "Описание — это детализированное изображение объекта, сцены или явления с акцентом на визуальные или сенсорные характеристики.\n\n"
            "Примеры:\n"
            "Текст: 'Биология — наука, изучающая живые организмы и их взаимодействие с окружающей средой.'\n"
            "Ответ: определение\n\n"
            "Текст: 'Лес был густым, с высокими соснами, покрытыми мхом, и запахом хвои.'\n"
            "Ответ: описание\n\n"
            f"Текст: {{text}}\n\n"
            "Ответ: "
        )
    },
    {
        'name': 'Prompt_Original_reazon',
        'template': (
            "Классифицируй следующий текст как 'определение' или 'описание'. "
            "Определение объясняет значение термина или понятия, часто в формальном стиле. "
            "Описание рисует картину объекта, сцены o явления с деталями.\n\n"
            "Примеры:\n"
            "Текст: 'Философия — это наука, изучающая общие законы развития природы, общества и мышления.'\n"
            "Ответ: Классификация: определение. Причина: Текст формально объясняет значение термина 'философия'.\n\n"
            "Текст: 'Демократия — это форма правления, при которой власть принадлежит народу.'\n"
            "Ответ: Классификация: определение. Причина: Текст четко определяет понятие 'демократия'.\n\n"
            "Текст: 'Годовалые побеги утолщенные, короткие, темно-бурые, при сушке чернеющие, серовато-шерстисто-опушенные.'\n"
            "Ответ: Классификация: описание. Причина: Текст детализирует внешние характеристики побегов.\n\n"
            "Текст: 'Река сверкала под лучами солнца, а берега утопали в зелени.'\n"
            "Ответ: Классификация: описание. Причина: Текст создает визуальную картину сцены.\n\n"
            f"Текст: {{text}}\n\n"
            "Ответ: Классификация: [определение/описание]. Причина: [объяснение, почему выбрана эта классификация]."
        )
    },
    {
        'name': 'Prompt_Simple_reazon',
        'template': (
            "Определи, является ли текст 'определением' или 'описанием'. "
            "Определение объясняет, что такое объект или понятие. "
            "Описание описывает внешний вид или характеристики объекта.\n\n"
            f"Текст: {{text}}\n\n"
            "Ответ: Классификация: [определение/описание]. Причина: [объяснение, почему выбрана эта классификация]."
        )
    },
    {
        'name': 'Prompt_Detailed_reazon',
        'template': (
            "Классифицируй текст как 'определение' или 'описание'. "
            "Определение — это формальное объяснение термина, понятия o proceso, generalmente con estructura clara. "
            "Описание — es una imagen detallada de un objeto, escena o fenómeno, centrándose en las características visuales o sensoriales.\n\n"
            "Ejemplos:\n"
            "Текст: 'Биология — наука, изучающая живые организмы и их взаимодействие с окружающей средой.'\n"
            "Respuesta: Классификация: определение. Причина: Текст формально объясняет, что такое биология.\n\n"
            "Текст: 'Лес был густым, с высокими соснами, покрытыми мхом, и запахом хвои.'\n"
            "Respuesta: Классификация: описание. Причина: Текст создает una imagen visual y sensorial del bosque.\n\n"
            f"Текст: {{text}}\n\n"
            "Ответ: Классификация: [определение/описание]. Причина: [объяснение, почему выбрана эта классификация]."
        )
    }
]

# (Your zero_shot_models and datasets definitions go here, as you provided them)
zero_shot_models = [
    {'model': 'gpt2',
     'name': 'gpt2',
     'type': 'gpt'},
]
datasets = [
    {
        'path1': '../dataset/Первый_жанр_исходная.txt',
        'path2': '../dataset/Второй_жанр_исходная.txt',
        'name': '1.Первый жанр исходная выборка',
        'type': 'pos',
        'freq': None
    },
]

# --- START OF MODIFIED FUNCTIONS ---

def zero_shot_classification_batch(texts, model, tokenizer, prompts, device='cuda' if torch.cuda.is_available() else 'cpu', batch_size=4, max_new_tokens=100):
    """Zero-shot classification in batches, generative, including reasoning in Russian."""
    results_per_prompt = {prompt['name']: [] for prompt in prompts}

    for prompt in prompts:
        prompt_name = prompt['name']
        prompt_template = prompt['template']

        predictions = []

        for i in tqdm(range(0, len(texts), batch_size), desc=f"Clasificando con {prompt_name}"):
            batch_texts = texts[i:i + batch_size]
            batch_prompts = [prompt_template.format(text=text) for text in batch_texts]
            
            # Tokenize the prompts
            inputs = tokenizer(batch_prompts, return_tensors='pt', padding=True, truncation=True, max_length=512).to(device)

            with torch.no_grad():
                # Generate text based on the prompt
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False, # Set to True for more varied output, False for deterministic
                    pad_token_id=tokenizer.eos_token_id, # Ensure pad_token_id is correctly set for generation
                    # Add other generation parameters as needed, e.g., temperature, top_k, top_p
                )

            # Decode the generated text, excluding the input prompt
            # We need to slice the output to get only the newly generated part
            generated_full = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
            
            # Extract only the generated completion by removing the original prompt
            generated_completions = []
            for j, gen_text in enumerate(generated_full):
                # Find where the prompt ends in the generated text
                # It's important to find the exact match of the prompt template used
                prompt_len = len(batch_prompts[j])
                # Check if the generated text starts with the prompt
                if gen_text.startswith(batch_prompts[j]):
                    completion = gen_text[len(batch_prompts[j]):].strip()
                else:
                    # Fallback for cases where the model might slightly alter the prompt or not include it fully
                    # This might happen with certain models or tokenization strategies
                    completion = gen_text # Use full generated text if prompt isn't clearly prefixed
                generated_completions.append(completion)

            for gen_text in generated_completions:
                # Regex to extract classification and reason
                # IMPORTANT: Corrected regex for 'определение'
                clasificacion_match = re.search(r'Классификация:\s*(определение|описание)', gen_text, re.IGNORECASE) # Added re.IGNORECASE
                razon_match = re.search(r'Причина:\s*(.*)', gen_text, re.MULTILINE)

                clasificacion = clasificacion_match.group(1).lower() if clasificacion_match else "не найдена"
                razon = razon_match.group(1).strip() if razon_match else "причина не найдена"

                predictions.append({'classification': clasificacion, 'reason': razon})

        results_per_prompt[prompt_name] = predictions

    return results_per_prompt

def evaluate_zero_shot(test_loader, model_name, prompts, batch_size=8):
    """Evaluates the model with zero-shot, now including generated reasons."""
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    
    # Ensure pad_token_id is set in model config if it's not by default
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
    model.config.pad_token_id = tokenizer.pad_token_id 
    model.eval()

    texts = [text for text, _ in test_loader]
    true_labels_raw = [label for _, label in test_loader]
    # Map true labels to 0 (описание) and 1 (определение)
    # Assuming original labels are such that 0 corresponds to description and 1 to definition
    true_labels_binary = [0 if l == 0 else 1 for l in true_labels_raw] # Corrected from `l != 1` to `l == 0` for clarity

    results_per_prompt = zero_shot_classification_batch(
        texts,
        model,
        tokenizer,
        prompts,
        device,
        batch_size
    )

    metrics_per_prompt = {}
    for prompt_name, raw_predictions in results_per_prompt.items():
        # Convert string classifications to binary for metric calculation
        pred_labels_binary = []
        extracted_reasons = [] # To store reasons for output
        for p in raw_predictions:
            classification = p['classification']
            reason = p['reason']
            extracted_reasons.append(reason) # Store the reason

            # IMPORTANT: Corrected classification string check (lowercase 'к')
            if classification == "описание":
                pred_labels_binary.append(0) # 0 for 'описание'
            elif classification == "определение":
                pred_labels_binary.append(1) # 1 for 'определение'
            else:
                pred_labels_binary.append(-1) # Handle 'не найдена' or other unexpected outputs
        
        # Filter out invalid predictions for metric calculation
        valid_indices = [i for i, label in enumerate(pred_labels_binary) if label in [0, 1]]
        if not valid_indices:
            print(f"Warning: No valid predictions for prompt '{prompt_name}'. Skipping metrics.")
            metrics_per_prompt[prompt_name] = {
                'accuracy': 0.0, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'confusion_matrix': [[0,0],[0,0]],
                'predictions_with_reasons': [] # Store raw predictions for inspection
            }
            continue


        filtered_true_labels = [true_labels_binary[i] for i in valid_indices]
        filtered_pred_labels = [pred_labels_binary[i] for i in valid_indices]

        accuracy = accuracy_score(filtered_true_labels, filtered_pred_labels)
        precision, recall, f1, _ = precision_recall_fscore_support(
            filtered_true_labels,
            filtered_pred_labels,
            average='binary',
            zero_division=0
        )
        cm = confusion_matrix(filtered_true_labels, filtered_pred_labels, labels=[0, 1])

        metrics_per_prompt[prompt_name] = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'confusion_matrix': cm,
            'predictions_with_reasons': raw_predictions # Store the full predictions (classification + reason)
        }

    return metrics_per_prompt

# --- END OF MODIFIED FUNCTIONS ---

# Your auxiliary classes (TextDataset, SentenceCleaner, DataLoader_raw) and functions
# (tokenizar_fuction, cargar_datos) remain the same.
class TextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item
class SentenceCleaner:
    def __init__(self, min_length_threshold=6):
        """Inicializa el procesador con un tokenizador y un umbral de longitud mínima."""
        self.min_length_threshold = min_length_threshold
    
    def _clean_and_split_text(self, texto):
        """
        Limpia y divide un texto en oraciones procesadas.
        
        Args:
            texto (str): Texto crudo a procesar.
        
        Returns:
            list: Lista de oraciones limpias.
        """
        # Paso 1: Marcar puntos para evitar fusiones no deseadas
        texto_limpio = re.sub(r'\.,', '. Ok999999999 ,', texto)
        texto_limpio = re.sub(r'\.;', '. Ok999999999 ', texto_limpio)
        texto_limpio = re.sub(r'\. ([a-zа-я])', r'. Ok999999999 \1', texto_limpio)
        texto_limpio = re.sub(r'(\w)([А-Я])', r'\1. \2', texto_limpio)
        
        # Paso 2: Eliminar corchetes y dividir en oraciones
        texto_sin_corchetes = re.sub(r'\[.*?\]', '', texto_limpio).strip()
        oraciones = [oracion.text for oracion in sentenize(texto_sin_corchetes)]
        
        # Paso 3: Restaurar espacios y limpiar marcadores temporales
        oraciones_limpias = [re.sub(r'\s*Ok999999999', ' ', oracion).strip() for oracion in oraciones]
        
        return oraciones_limpias
    
    def _process_sentences(self, datos, target_list):
        for texto, etiqueta in datos:
            oraciones = self._clean_and_split_text(texto)
            for oracion in oraciones:
                tokens = tokenizar_fuction(oracion)
                if len(tokens) >= self.min_length_threshold:
                    target_list.append((oracion, etiqueta))
    
    def __call__(self, data_raw):
        datos_procesados = []
        self._process_sentences(data_raw, datos_procesados)
        
        return {
            'datos_procesados': datos_procesados,
        }
class DataLoader_raw:
    def __init__(self, path1, path2):
        self.path1 = path1
        self.path2 = path2
    
    def __call__(self):
        datos_genero1 = cargar_datos(self.path1, 0)
        datos_genero2 = cargar_datos(self.path2, 1)
        datos = datos_genero1 + datos_genero2
        
        return {
            'datos_raw': datos,
        }
def tokenizar_fuction(texto):
    """Tokeniza el texto eliminando puntuación y convirtiendo a minúsculas."""
    texto_preprocesado = re.sub(r'[^\w\s]', '', texto.lower())
    tokens = nltk.word_tokenize(texto_preprocesado)
    return tokens
def cargar_datos(archivo, etiqueta):
    with open(archivo, 'r', encoding='utf-8') as f:
        textos = f.readlines()
    return [(texto.strip(), etiqueta) for texto in textos]


def main():
    results = []
    
    for model_info in zero_shot_models: # Renamed 'model' to 'model_info' for clarity
        for dataset in datasets:
            print(f"\n--- Running for Model: {model_info['name']}, Dataset: {dataset['name']} ---")
            ppl = PipelineCommon([
                (DataLoader_raw(dataset['path1'], dataset['path2']), [], {'datos_raw': 'datos_raw'}),
                (SentenceCleaner(), ['datos_raw'], {'datos_procesados': 'datos_procesados'}),
            ])
            result = ppl()
            test_loader = result['datos_procesados']
            
            metrics_per_prompt = evaluate_zero_shot(
                test_loader,
                model_info['model'], # Use model_info['model'] to get the Hugging Face ID
                prompts,
                batch_size=16
            )
            
            for prompt_name, metrics in metrics_per_prompt.items():
                results.append({
                    'dataset': dataset['name'],
                    'model': model_info['name'],
                    'prompt': prompt_name,
                    'metrics': metrics
                })
            
            # Clear GPU cache after each model-dataset run
            torch.cuda.empty_cache()
            gc.collect()
    
    print("\n--- Final Results ---")
    for res in results:
        print(f"Dataset: {res['dataset']}, Model: {res['model']}, Prompt: {res['prompt']}")
        print(f"Métricas: {res['metrics']['accuracy']:.4f} (Accuracy), {res['metrics']['f1']:.4f} (F1-Score)")
        print(f"Matriz de Confusión:\n{res['metrics']['confusion_matrix']}")
        
        # Print a few example predictions with reasons for 'reazon' prompts
        if 'reazon' in res['prompt'].lower() and 'predictions_with_reasons' in res['metrics']:
            print("Ejemplos de clasificaciones con razones:")
            # Limit to 5 examples for brevity
            for i, pred_info in enumerate(res['metrics']['predictions_with_reasons'][:5]):
                original_text = texts[i] if i < len(texts) else "Original text not available"
                print(f"  Texto: \"{original_text}\"")
                print(f"  Clasificación Predicha: {pred_info['classification'].capitalize()}")
                print(f"  Razón: {pred_info['reason']}\n")
        print("-" * 50) # Separator for readability

if __name__ == "__main__":
    main()


--- Running for Model: gpt2, Dataset: 1.Первый жанр исходная выборка ---


Using pad_token, but it is not set yet.
Clasificando con Prompt_Original:   0%|          | 0/54 [00:00<?, ?it/s]2025-06-17 12:54:56.059126: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
